# Binary Classification U-Net for Brain Tumor Detection

- The model outputs two logits per input, which are converted via Softmax into probabilities for the classes ["no tumor", "tumor"].

In [ ]:
from src.brain_tumor_semantic_segmentation.data import load_mri_dataframe, get_dataloader_binarytransformed


df = load_mri_dataframe()
cls_train_loader, cls_val_loader = get_dataloader_binarytransformed(df, batch_size=8, augment=False)

print(f"Train Samples: {len(cls_train_loader.dataset)} | Val Samples: {len(cls_val_loader.dataset)}")

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import vgg16, VGG16_Weights

class VGG16(nn.Module):
    """
    VGG16-based classification model for brain tumor detection.
    
    Outputs:
        pre-Softmax scores (logits) for no tumor/tumor classes
    """
    def __init__(self, dropout_prob=0.3, freeze_features=True):
        super().__init__()
        base = vgg16(weights=VGG16_Weights.DEFAULT)
        self.features = base.features
        self.avgpool  = base.avgpool

        if freeze_features:
            for p in self.features.parameters():
                p.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_prob),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_prob),
   
            nn.Linear(256, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        logits = self.classifier(x)

        return logits

In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
  

model = VGG16(freeze_features=True)
print("Number of Parameters (Classification, no augment):", sum(p.numel() for p in model.parameters()))
model.to(device)
print(f"Device: {device}.")

In [ ]:
import torch.optim as optim
from src.brain_tumor_semantic_segmentation.train_generalized import train


trained_model, training_results = train(
    model,
    cls_train_loader,
    cls_val_loader,
    device,
    epochs=60,
    lr=1e-3,
    optimizer_class=optim.Adam,
    loss_fn = nn.CrossEntropyLoss,
    task="classification",
    save_name="binary_classification_model"
)



In [ ]:
from src.brain_tumor_semantic_segmentation.evaluate import evaluate_classification 
class_names = ["no Tumor", "Tumor"]
evaluate_classification(
    results=training_results,
    num_batches=2,
    class_names=class_names
)